---
title : "Validator Agent"
execute:
    output: true
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

De validator Agent van 'Talk with FEWS' heeft verschillende tools tot zijn beschikking.
Het taalmodel kan deze zelf invullen, maar hier wordt per tool een voorbeeld gegeven van de in en output.

### Voorbereidingen

We gaan als eerste een context object definieren. Het context object bevat informatie welke de tools gebruik van kunnen maken. Voor de validator Agent is de verwijzing naar de locatie van de FEWS XSD schema's het meest relevant binnen dit context object.

In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

C:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\.pixi\envs\default\Lib\site-packages\pydantic\fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


## Validate XML snippet

Delft FEWS configuratie is voor een groot gedeelte opgebouwd rond XML bestanden. Deze XML bestanden moeten voldoen aan een bepaalde structuur. Deze structuur wordt bepaald door de bijbehorende XSD schema.

Laten we beginnen met een XML snippet van een time `series` definitie. Er mist een element in deze snippet welke we hopen te detecteren door onze validatie tool.

In [2]:
xml_snippet = """
<series>
    <header>
        <type>instantaneous</type>
        
        <parameterId>WaterLevel</parameterId>
        <timeStep unit="second" multiplier="3600"/>
        <startDate date="2006-08-23" time="17:00:00"/>
        <endDate date="2006-08-24" time="00:00:00"/>
        <missVal>-8888.0</missVal>
    </header>
    <event date="2006-08-23" time="17:00:00" value="8.66"/>
</series>
"""

Nu we onze XML snippet hebben ingevuld, gaan we hiermee de tool op de juiste manier aanroepen. Voor een taalmodel is het van belang dat het toegang heeft tot voldoende informatie om de tool op de juiste manier aan te roepen.

Dit doen wij via zogenaamd *model*: een soort sjabloon waarin je vastlegt welke gegevens er nodig zijn en welk type ze moeten hebben. Voordat de tool aangeroepen wordt wordt er gecontroleerd of de aangeleverde informatie voldoet aan de regels van het model.

Voor hetgeen uit de tool komt, hebben we ook een sjabloon met de gegevens die er uit de tool komen. Op deze manier heeft een taalmodel inzichtelijk, zonder de tool aan te roepen, wat de tool doet, op welke manier het ingevuld moet worden en wat er uit de tool terug komt.

Laten we de validate XML tool aanroepen met onze XML snippet en het bijbehorende XSD schema. 

```python
XMLValidationInput(
    xml_content=xml_snippet, 
    schema_option=FEWSSchemaOption.PI_TIMESERIES
)
```

In [3]:
from talk_with_fews.tools.validator.validate_xml import validate_xml
from talk_with_fews.models.validate_xml import XMLValidationOutput, XMLValidationInput
from talk_with_fews.models.fews_schema_options import FEWSSchemaOption

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_option=FEWSSchemaOption.PI_TIMESERIES
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

C:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\src\talk_with_fews\tools\validator\validate_xml.py:167: XMLSchemaIncludeWarning: Include schema failed: can't access to resource 'file:///C:/Data/Python/PR5473.10_Talk_with_Fews/talk-with-fews/data/fews_schemas/fewsdocs_deltares_nl/pi-schemas/pi_sharedtypes.xsd': [WinError 2] The system cannot find the file specified: 'C:\\Data\\Python\\PR5473.10_Talk_with_Fews\\talk-with-fews\\data\\fews_schemas\\fewsdocs_deltares_nl\\pi-schemas\\pi_sharedtypes.xsd'
  schema = xmlschema.XMLSchema(str(full_path), validation="lax")


Laten we afdrukken wat het resultaat is van de XML snippet validatie

In [4]:
print(format_as_xml(xml_validation_output))

<is_valid>True</is_valid>
<errors />
<warnings />
<schema_location>data/fews_schemas/fewsdocs_deltares_nl/pi-schemas/pi_timeseries.xsd</schema_location>


De tool resultaat laat zien dat de XML snippet niet valide is, en dat er eerst een `locationId` wordt verwacht, waar het nu een `parameterId` aangeboden krijgt.

Wanneer de XML snippet opnieuw invullen, inclusief een `locationId` krijgen we het volgende.

In [5]:
xml_snippet = """
<series>
    <header>
        <type>instantaneous</type>
        <locationId>1234567890</locationId>
        <parameterId>WaterLevel</parameterId>
        <timeStep unit="second" multiplier="3600"/>
        <startDate date="2006-08-23" time="17:00:00"/>
        <endDate date="2006-08-24" time="00:00:00"/>
        <missVal>-8888.0</missVal>
    </header>
    <event date="2006-08-23" time="17:00:00" value="8.66"/>
</series>
"""

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_option=FEWSSchemaOption.PI_TIMESERIES
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

print(format_as_xml(xml_validation_output))

<is_valid>True</is_valid>
<errors />
<warnings />
<schema_location>data/fews_schemas/fewsdocs_deltares_nl/pi-schemas/pi_timeseries.xsd</schema_location>


Het resultaat is nu dat de XML snippet succesvol valideert zonder errors or warnings.

Hieronder nog een voorbeeld dat laat zien dat deze partiele validatie ook nog goed gaat voor XML snippets *dieper* in een XML bestand, exclusief een directe verwijzing naar zijn directe *parent* element.

In [6]:
# xml snippet with indirect parent to root with missing nested elements
xml_snippet = """
	<general>
		<exportType>SomeValidExportType</exportType>
		<folder>MyExportFolder</folder>
		<exportFileName>
			<prefix>
				<timeZeroFormattingString>yyyyMMddHHmm</timeZeroFormattingString>
			</prefix>
		</exportFileName>
		<idMapId>MyIdMap</idMapId>
		<forecastSelectionPeriod end="0" unit="day"/>
	</general>
"""

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_option=FEWSSchemaOption.TIME_SERIES_EXPORT_RUN
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

print(format_as_xml(xml_validation_output))

<is_valid>True</is_valid>
<errors />
<warnings />
<schema_location>data/fews_schemas/fewsdocs_deltares_nl/timeSeriesExportRun.xsd</schema_location>


C:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\src\talk_with_fews\tools\validator\validate_xml.py:167: XMLSchemaIncludeWarning: Include schema failed: can't access to resource 'file:///C:/Data/Python/PR5473.10_Talk_with_Fews/talk-with-fews/data/fews_schemas/fewsdocs_deltares_nl/sharedTypes.xsd': [WinError 2] The system cannot find the file specified: 'C:\\Data\\Python\\PR5473.10_Talk_with_Fews\\talk-with-fews\\data\\fews_schemas\\fewsdocs_deltares_nl\\sharedTypes.xsd'
  schema = xmlschema.XMLSchema(str(full_path), validation="lax")


Binnen de error beschrijving staat als onderdeel van de `xpath` de route welke de validator heeft afgelegd om deze snippet binnen het XSD schema te valideren. 

Het heeft dus gevonden dat onze snippet onderdeel is van de `general` element, en dat via `export` teruggeleid heeft naar de `timeSeriesExportRun`, de root element van dit XSD schema.

## Genereer XML snippet Template

Deze tool maakt het mogelijk om een template op te zetten voor een bepaald gedeelte binnen je XML bestand . 

Hieronder laten we zien hoe we een template kunnen laten genereren voor de `general` element binnen het schema van de timeSeriesExportRun.

In [7]:
# partial xml template generation

from talk_with_fews.tools.validator.generate_xml_template import generate_xml_template
from talk_with_fews.models.generate_xml_template import (
    XMLTemplateInput,
    XMLTemplateOutput,
)
from talk_with_fews.models.fews_schema_options import FEWSSchemaOption

xml_template_input = XMLTemplateInput(
    schema_option=FEWSSchemaOption.TIME_SERIES_EXPORT_RUN, target_element="general"
)
xml_template_output: XMLTemplateOutput = await generate_xml_template(
    ctx=ctx, input=xml_template_input
)

print(xml_template_output.template)

<general>
  <exportTypeStandard>example_value</exportTypeStandard>
  <exportType>example_value</exportType>
  <serializerClassName>example_value</serializerClassName>
  <folder/>
  <exportFileName>
    <name>example_value</name>
    <useExternalLocationIdAsName>true</useExternalLocationIdAsName>
  </exportFileName>
  <serverUrl>example_value</serverUrl>
  <user>example_value</user>
  <password>example_value</password>
  <encryptedPassword>example_value</encryptedPassword>
  <exportMissingValue>1.0</exportMissingValue>
  <exportMissingValueString>example_value</exportMissingValueString>
  <dataFeedId/>
  <disableDataFeedInfo/>
  <columnSeparator/>
  <decimalSeparator/>
</general>


C:\Data\Python\PR5473.10_Talk_with_Fews\talk-with-fews\src\talk_with_fews\tools\validator\generate_xml_template.py:219: XMLSchemaIncludeWarning: Include schema failed: can't access to resource 'file:///C:/Data/Python/PR5473.10_Talk_with_Fews/talk-with-fews/data/fews_schemas/fewsdocs_deltares_nl/sharedTypes.xsd': [WinError 2] The system cannot find the file specified: 'C:\\Data\\Python\\PR5473.10_Talk_with_Fews\\talk-with-fews\\data\\fews_schemas\\fewsdocs_deltares_nl\\sharedTypes.xsd'
  schema = xmlschema.XMLSchema(str(full_path), validation="lax")


Op het moment wanneer de `target_element` niet goed ingevuld wordt, retourneren we een `ModelRetry`, waarin we een suggestie aanleveren gebasseerd op wat geprobeerd is.

Hier is het `target_element` ingevuld als `genral` in plaats van `general`:

In [8]:
try:
    xml_template_input = XMLTemplateInput(
        schema_option=FEWSSchemaOption.TIME_SERIES_EXPORT_RUN, target_element="genral"
    )
    xml_template_output: XMLTemplateOutput = await generate_xml_template(
        ctx=ctx, input=xml_template_input
    )

    print(xml_template_output.template)
except Exception as e:
    print(e)

Target element 'genral' not found in schema 'timeSeriesExportRun.xsd'.

Similar elements found:
  - general


Please try again with one of the suggested elements.

Tip: Use the 'get_available_elements' tool to get a complete list of available elements in this schema.


Zie dat een duidelijke `ModelRetry` teruggeeft met een suggestie voor het het taalmodel of Agent om de tool op een andere manier aan te roepen:

```bash
ModelRetry: Target element 'genral' not found in schema 'timeSeriesExportRun.xsd'.

Similar elements found:
  - general


Please try again with one of the suggested elements.
```

## Get available elements

Deze tool maakt het mogelijk om een lijst te krijgen van alle elementen die beschikbaar zijn in een bepaald schema.

Hieronder laten we zien hoe we een lijst kunnen krijgen van alle elementen die beschikbaar zijn in het schema van de `timeSeriesExportRun`.


In [9]:
from talk_with_fews.tools.validator.get_available_elements import get_available_elements
from talk_with_fews.models.get_available_elements import (
    GetAvailableElementsInput,
    GetAvailableElementsOutput,
)

get_available_elements_input = GetAvailableElementsInput(
    schema_option=FEWSSchemaOption.TIME_SERIES_EXPORT_RUN,
    limit=15,
    offset=0,
    target_element="general",
)
get_available_elements_output: GetAvailableElementsOutput = (
    await get_available_elements(ctx=ctx, input=get_available_elements_input)
)

print(format_as_xml(get_available_elements_output))

<schema_location>data/fews_schemas/fewsdocs_deltares_nl/timeSeriesExportRun.xsd</schema_location>
<root_element>{http://www.wldelft.nl/fews}timeSeriesExportRun</root_element>
<target_element>general</target_element>
<available_elements>
  <item>backupServerUrl</item>
  <item>binDir</item>
  <item>coldStartDataExportRelativeStartTime</item>
  <item>columnSeparator</item>
  <item>connectionTimeOutMillis</item>
  <item>convertAnnotationEnumerationPropertyValuesToLabels</item>
  <item>convertDatum</item>
  <item>convertValuesToEnumerationLabels</item>
  <item>creationTime</item>
  <item>dataFeedId</item>
  <item>dateTimePattern</item>
  <item>dbChangeViewPeriod</item>
  <item>decimalSeparator</item>
  <item>description</item>
  <item>disableDataFeedInfo</item>
</available_elements>
<total_count>57</total_count>
<limit>15</limit>
<offset>0</offset>
<has_more>True</has_more>
<next_offset>15</next_offset>


We zien dat er in totaal 93 elementen in de XSD schema bestaan binnen het target element `general`. En we laten er de eerste 15 zien, waarbij we tegelijk aangegeven dat er meer beschikbaar zijn. Op deze manier kan de tool er gepagineerd door heen stappen. Zonder gelijk heel veel terug te krijgen.